# Matrix 8b × A (natural, 4M/500)


In [ ]:
import json, shutil, subprocess, sys
from pathlib import Path
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('no GPU')
p = torch.cuda.get_device_properties(0)
print('gpu', p.name, round(p.total_memory / 1e9, 1), 'GB')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True)


In [ ]:
DATA_SRC = Path('/kaggle/input/tiki-matrix-data')
if not (DATA_SRC / 'units_A.parquet').exists():
    hits = sorted(Path('/kaggle/input').rglob('units_A.parquet'))
    assert hits, 'không thấy tiki-matrix-data'
    DATA_SRC = hits[0].parent
ROOT = Path('/kaggle/working/matrix_data')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
for name in ('A', 'B'):
    (ROOT / f'processed_{name}').mkdir(parents=True)
    shutil.copy2(DATA_SRC / f'units_{name}.parquet', ROOT / f'processed_{name}/units.parquet')
    shutil.copy2(DATA_SRC / f'eval_{name}.parquet', ROOT / f'eval_{name}.parquet')
    shutil.copy2(DATA_SRC / f'meta_{name}.json', ROOT / f'meta_{name}.json')
CODE = Path('/kaggle/working/code')
CODE.mkdir(exist_ok=True)
for name in ('train_weight_quota_gpu.py', 'quota_sampling.py', 'tpu_weight_train.py'):
    shutil.copy2(DATA_SRC / name, CODE / name)

HIDDEN = 2560
MODEL = None
for cfg in Path('/kaggle/input').rglob('config.json'):
    try:
        d = json.loads(cfg.read_text(encoding='utf-8'))
    except Exception:
        continue
    if d.get('hidden_size') == HIDDEN and d.get('num_hidden_layers') in (36,):
        MODEL = str(cfg.parent)
        break
assert MODEL, f'không thấy model hidden={HIDDEN}'
print('DATA', DATA_SRC, '| MODEL', MODEL)


In [ ]:
import pandas as pd
ev = pd.read_parquet(DATA_SRC / 'eval_A.parquet')
for ds in ('A', 'B'):
    U = pd.read_parquet(DATA_SRC / f'units_{ds}.parquet')
    overlap = set(ev.product_id) & set(U.product_id)
    tok = float(U.eff_tokens.sum())
    print(ds, 'rows', len(U), 'tokensM', round(tok/1e6, 3),
          'overlap', len(overlap), 'epochs@4M', round(4e6/tok, 3))
    assert not overlap, f'LEAK {ds}: {len(overlap)} products'
print('LEAK CHECK OK')


In [ ]:
DATASET = 'B'
CONFIG = 'balanced'
MODEL_LABEL = 'm4b'
OUT = Path('/kaggle/working/out_B_m4b_' + CONFIG)
if OUT.exists():
    shutil.rmtree(OUT)
cmd = [sys.executable, str(CODE / 'train_weight_quota_gpu.py'),
       '--dataset', DATASET, '--data-root', str(ROOT), '--out', str(OUT),
       '--model', MODEL, '--seed', '42', '--tokens', '4000000', '--steps', '500',
       '--configs', CONFIG, '--micro-batch', str(4)]
print('RUN', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)

# đổi tên hyp để scoring đọc được: hyp_<model>_<dataset>_<config>_seed42.parquet
src = OUT / f'hyp_{CONFIG}_seed42.parquet'
dst = OUT / f'hyp_{MODEL_LABEL}_{DATASET.lower()}_{CONFIG}_seed42.parquet'
src.rename(dst)
print('HYP', dst, dst.stat().st_size)
print((OUT / 'summary.json').read_text())
